# Inspect the fitted SDE-GAN

Load saved models to check solver accuracy, compare volatility, and plot
20-session scenarios. Use the `sandbox/.venv` kernel and run from the top.

Set `RUN_FOLDER` to the experiment folder, or `""` for the original baseline.
The weight-30 trial has only seed 0, so keep `RUN_SEEDS = [0]`. This notebook
uses training and validation data; test data remain reserved for final evaluation.

In [ ]:
from pathlib import Path
from dataclasses import replace, asdict
import json
import os
import sys

os.environ["JAX_PLATFORMS"] = "cpu"  # Restart the kernel if JAX already initialized elsewhere.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "sandbox/4stocks_experiment/sde_gan.py").is_file())
experiment_dir = root / "sandbox/4stocks_experiment"
sys.path.insert(0, str(experiment_dir)) if str(experiment_dir) not in sys.path else None
from sde_gan import SDEGAN

RUN_FOLDER = "temporal_penalty_30"  # Use "" for the original baseline.
runs_dir = experiment_dir / "results/sde_gan" / RUN_FOLDER
RUN_SEEDS = [0]  # The weight-30 trial trained seed 0 only.
PLOT_SEED = 0  # Chosen in advance, not the run with the most attractive plots.
N_PATHS = 1000
SAMPLE_SEED = 100
BOOTSTRAP_REPEATS = 300
BOOTSTRAP_SEED = 2026
COARSE_DT = 0.25
FINE_DT = 0.125
out_dir = runs_dir / "inspection"
out_dir.mkdir(parents=True, exist_ok=True)
models, metadata, selected = {}, {}, []
for seed in RUN_SEEDS:
    run_dir = runs_dir / f"seed_{seed}"
    models[seed] = SDEGAN.load(run_dir)
    metadata[seed] = json.loads((run_dir / "preprocessing.json").read_text())
    selected.append({"seed": seed, **models[seed].run_metadata,
                     "temporal_penalty_weight": models[seed].config.temporal_penalty_weight})
reference_metadata = metadata[RUN_SEEDS[0]]
if any(m != reference_metadata for m in metadata.values()):
    raise ValueError("Runs use different preprocessing; compare matching data separately")
STOCKS = reference_metadata["stock_order"]
display(pd.DataFrame(selected).set_index("seed")[["selected_step", "validation_score", "kernel_bandwidth", "temporal_penalty_weight"]])
print(f"Python: {sys.executable}; {N_PATHS} paths per seed and solver step")
print(f"Checkpoints: {runs_dir}\nInspection outputs: {out_dir}")

## Restore ordinary returns

Undo the saved scaling and log transformation so returns can be read as
percentages. Compare against the same complete 20-session windows used in
preprocessing. Read observed values directly from CSVs to preserve exact zeros.
Overlapping windows share dates, so they are not independent market periods.

In [ ]:
def to_simple(paths: np.ndarray, meta: dict) -> np.ndarray:
    """Invert the saved scaler without changing the generated trajectories."""
    if paths.ndim != 3 or paths.shape[1:] != (20, 4) or not np.isfinite(paths).all():
        raise ValueError("Expected finite standardized paths with shape (n, 20, 4)")
    with np.errstate(over="raise", invalid="raise"):
        result = np.expm1(paths.astype(np.float64) * np.asarray(meta["training_std"])
                          + np.asarray(meta["training_mean"]))
    if not np.isfinite(result).all():
        raise ValueError("Nonfinite simple returns")
    return result


def observed_windows(split: str) -> np.ndarray:
    """Load original simple-return windows without bridging missing sessions."""
    frame = pd.read_csv(experiment_dir / "data" / f"{split}_returns.csv",
                        index_col="Date", parse_dates=["Date"]).loc[:, STOCKS]
    if not frame.index.is_unique or not frame.index.is_monotonic_increasing:
        raise ValueError(f"{split}: invalid date index")
    dates = pd.read_csv(experiment_dir / "data/window_dates.csv",
                        parse_dates=["start_date", "end_date"])
    dates = dates.loc[dates.split.eq(split)].sort_values("window_index")
    paths = np.stack([frame.loc[row.start_date:row.end_date].to_numpy()
                      for row in dates.itertuples()])
    if paths.shape[1:] != (20, 4) or not np.isfinite(paths).all():
        raise ValueError(f"{split}: incomplete observed windows")
    # Check that current CSVs and window dates still match the saved prepared data.
    with np.load(experiment_dir / "data/prepared_windows.npz", allow_pickle=False) as prepared:
        np.testing.assert_allclose(to_simple(prepared[f"{split}_windows"], reference_metadata),
                                   paths, rtol=1e-6, atol=1e-7)
    if len(paths) != reference_metadata["splits"][split]["retained"]:
        raise ValueError("Observed windows differ from checkpoint metadata")
    return paths


observed = {split: observed_windows(split) for split in ["train", "validation"]}
print("Observed shapes:", {k: v.shape for k, v in observed.items()})

## Compare solver steps

Generate paths at steps 0.25 and 0.125 using the same model and randomness.
Changing only the step isolates numerical differences. Saved model files stay
unchanged.

In [ ]:
coarse_simple, fine_simple, path_errors = {}, {}, []
for seed, model in models.items():
    if not np.isclose(model.config.generator_dt, COARSE_DT):
        raise ValueError("This comparison expects a checkpoint trained with step 0.25")
    fine = SDEGAN.load(runs_dir / f"seed_{seed}")
    fine.config = replace(fine.config, generator_dt=FINE_DT)
    coarse_config, fine_config = asdict(model.config), asdict(fine.config)
    assert [k for k in coarse_config if coarse_config[k] != fine_config[k]] == ["generator_dt"]
    print(f"Sampling seed {seed}: steps {COARSE_DT} and {FINE_DT}", flush=True)
    coarse = model.sample(n_paths=N_PATHS, seed=SAMPLE_SEED)
    refined = fine.sample(n_paths=N_PATHS, seed=SAMPLE_SEED)
    np.testing.assert_array_equal(coarse[:, 0], refined[:, 0])
    np.testing.assert_array_equal(refined[:17], fine.sample(n_paths=17, seed=SAMPLE_SEED))
    coarse_simple[seed] = to_simple(coarse, metadata[seed])
    fine_simple[seed] = to_simple(refined, metadata[seed])
    path_errors.append({"seed": seed, **dict(zip(STOCKS,
        np.sqrt(np.mean((fine_simple[seed] - coarse_simple[seed])**2, axis=(0, 1))) * 100))})
    np.savez_compressed(out_dir / f"samples_seed_{seed}.npz",
                        coarse_standardized_log=coarse, fine_standardized_log=refined)
path_errors = pd.DataFrame(path_errors).set_index("seed")
print("Paired daily-return RMSE (percentage points):")
display(path_errors)

## Check whether solver differences matter

Resample paired generated paths to estimate simulation uncertainty in means,
volatility, and portfolio losses. Compare step differences with that uncertainty
to assess whether 0.25 is accurate enough for these summaries.

Values are percentage points; differences are fine minus coarse. The intervals
cover simulation noise, not uncertainty about training or observed market data.

In [ ]:
def summary_vector(simple: np.ndarray) -> np.ndarray:
    """Summarize moments and exact compounded simple portfolio returns."""
    flat = simple.reshape(-1, 4)
    portfolio_terminal = np.prod(1 + simple.mean(axis=2), axis=1) - 1
    return np.r_[flat.mean(axis=0), flat.std(axis=0, ddof=1),
                 np.quantile(portfolio_terminal, 0.05)] * 100


metric_names = [f"mean: {s}" for s in STOCKS] + [f"daily_std: {s}" for s in STOCKS] + ["portfolio_terminal_q05"]
sensitivity_tables = []
for seed in RUN_SEEDS:
    a, b = coarse_simple[seed], fine_simple[seed]
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    coarse_boot, delta_boot = [], []
    for _ in range(BOOTSTRAP_REPEATS):
        indices = rng.integers(0, len(a), size=len(a))
        av = summary_vector(a[indices])
        coarse_boot.append(av)
        delta_boot.append(summary_vector(b[indices]) - av)
    av, bv = summary_vector(a), summary_vector(b)
    se = np.std(coarse_boot, axis=0, ddof=1)
    interval = np.quantile(delta_boot, [0.025, 0.975], axis=0)
    table = pd.DataFrame({"seed": seed, "metric": metric_names,
        "coarse_pp": av, "fine_pp": bv, "difference_pp": bv-av,
        "coarse_mc_se_pp": se, "paired_delta_lower_pp": interval[0],
        "paired_delta_upper_pp": interval[1],
        "larger_than_mc95": np.abs(bv-av) > 1.96 * se})
    sensitivity_tables.append(table)
sensitivity = pd.concat(sensitivity_tables, ignore_index=True).set_index(["seed", "metric"])
sensitivity.to_csv(out_dir / "solver_sensitivity.csv")
display(sensitivity.round(6))
print(f"{int(sensitivity.larger_than_mc95.sum())} of {len(sensitivity)} summary changes exceed the MC review heuristic.")
print("Review magnitudes and the paired path plots. Step 0.25 remains the working default for the checked summaries.")
# Independent arithmetic check for portfolio compounding and metric units.
fixture = np.zeros((2, 20, 4))
fixture[:, 0, 0] = 0.10
np.testing.assert_allclose(summary_vector(fixture)[-1], 2.5)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, constrained_layout=True)
for j, ax in enumerate(axes.flat):
    ax.plot(np.arange(1, 21), coarse_simple[PLOT_SEED][0, :, j] * 100, label="step 0.25")
    ax.plot(np.arange(1, 21), fine_simple[PLOT_SEED][0, :, j] * 100, "--", label="step 0.125")
    ax.set(title=STOCKS[j], xlabel="Trading session", ylabel="Simple daily return (%)")
    ax.legend()
fig.suptitle(f"Same initial noise and Brownian realization: seed {PLOT_SEED}, path 0")
fig.savefig(out_dir / "coupled_solver_paths.png", dpi=150)
plt.show()
plt.close(fig)

## Compare volatility

Compare daily return spread with training and validation data. The CDFs show
what fraction of 20-session windows have volatility at or below each value,
revealing whether the model produces enough quiet and turbulent periods.
Volatility is reported in percent per session.

In [ ]:
datasets = {**observed, **{f"generated seed {s}": a for s, a in coarse_simple.items()}}
volatility = pd.DataFrame({name: a.reshape(-1, 4).std(axis=0, ddof=1) * 100
                           for name, a in datasets.items()}, index=STOCKS).T
volatility.index.name = "dataset"
volatility.to_csv(out_dir / "daily_volatility_pct.csv")
display(volatility.round(3))
fig, ax = plt.subplots(figsize=(13, 6), constrained_layout=True)
volatility.T.plot.bar(ax=ax)
ax.set(title="Observed and generated daily volatility", ylabel="Daily standard deviation (%)", xlabel="Stock")
ax.tick_params(axis="x", rotation=15)
fig.savefig(out_dir / "daily_volatility.png", dpi=150)
plt.show()
plt.close(fig)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for j, ax in enumerate(axes.flat):
    for name, paths in datasets.items():
        values = np.sort(paths[:, :, j].std(axis=1, ddof=1) * 100)
        ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=name)
    ax.set(title=STOCKS[j], xlabel="Within-window daily standard deviation (%)", ylabel="Empirical CDF")
    ax.legend(fontsize=8)
fig.savefig(out_dir / "window_volatility.png", dpi=150)
plt.show()
plt.close(fig)

## Plot generated scenarios

Plot five joint return scenarios and compound them into stock indices starting
at 100. Matching colors identify the same scenario across stocks. The shaded
band spans the 5th–95th percentiles at each session, showing the model's sampled
spread.

In [ ]:
scenario_returns = coarse_simple[PLOT_SEED]
HORIZON = scenario_returns.shape[1]
sessions = np.arange(1, HORIZON + 1)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, constrained_layout=True)
for j, ax in enumerate(axes.flat):
    for i in range(5):
        ax.plot(sessions, scenario_returns[i, :, j] * 100, label=f"Scenario {i + 1}")
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set(title=STOCKS[j], xlabel="Trading session", ylabel="Simple daily return (%)")
    ax.legend(fontsize=8)
fig.suptitle(f"{HORIZON}-session joint return scenarios; selected seed {PLOT_SEED} checkpoint")
fig.savefig(out_dir / "generated_returns.png", dpi=150)
plt.show()
plt.close(fig)
indexed = 100 * np.concatenate([np.ones((len(scenario_returns), 1, 4)),
                               np.cumprod(1 + scenario_returns, axis=1)], axis=1)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, constrained_layout=True)
for j, ax in enumerate(axes.flat):
    lo, mid, hi = np.quantile(indexed[:, :, j], [0.05, 0.5, 0.95], axis=0)
    ax.fill_between(np.arange(HORIZON + 1), lo, hi, alpha=0.18, color="grey", label="5–95% scenario band")
    for i in range(5):
        ax.plot(np.arange(HORIZON + 1), indexed[i, :, j], alpha=0.8)
    ax.plot(np.arange(HORIZON + 1), mid, color="black", linestyle="--", label="Median")
    ax.set(title=STOCKS[j], xlabel="Trading session", ylabel="Stock index (initial value 100)")
    ax.legend(fontsize=8)
fig.savefig(out_dir / "generated_stock_indices.png", dpi=150)
plt.show()
plt.close(fig)
np.savez_compressed(out_dir / "illustrated_scenarios.npz", simple_returns=scenario_returns,
                    indexed_stock_paths=indexed)
(out_dir / "inspection.json").write_text(json.dumps({
    "run_folder": RUN_FOLDER, "run_seeds": RUN_SEEDS, "plot_seed": PLOT_SEED, "sample_seed": SAMPLE_SEED,
    "n_paths": N_PATHS, "horizon": HORIZON, "coarse_dt": COARSE_DT, "fine_dt": FINE_DT,
    "bootstrap_repeats": BOOTSTRAP_REPEATS, "bootstrap_seed": BOOTSTRAP_SEED,
    "stock_order": STOCKS, "preprocessing": reference_metadata,
}, indent=2, allow_nan=False) + "\n")
print(f"Tables, samples, settings, and plots saved to {out_dir}")